In [71]:
!pip install numpy pandas scikit-learn


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [72]:
import numpy as np
import pandas as pd
import sklearn as skl

# Reading Data & Data Preprocessing

In [73]:
ds = []
with open("semeval-tweets/twitter-dev-data.txt") as f:
  for x in f:
    ds.append(x.split("\t"))

df = pd.DataFrame(data = ds, columns = ["id", "sentiment", "tweet"])
df

,id,sentiment,tweet
0,262696992176304465,neutral,@Oirisheye Hey you! I'm gonna be in Dublin in ...
1,410734138242126311,positive,Literally so excited I'm going to a Sam Smith ...
2,237615985571058688,neutral,@WINDmobile Will there be an option to buy the...
3,090473590077188360,neutral,Our Little Ms. Philippines. 👏👏👏#littleMsPhilip...
4,450236582392850660,negative,"@AngryRaiderFan I know. This, TPP, expanded wa..."
...,...,...,...
1995,648550834532150634,positive,can it just be 5:30 tomorrow evening alREADY b...
1996,022324697041215693,neutral,Look for me on @GMA tomorrow at 7:34am talking...
1997,885111074317866321,neutral,@TheShannonBurke after @immandaschmidt's call ...
1998,691260170402024265,neutral,"""Taking Go Set a Watchman with me to the beach..."


In [74]:
# test set
dstest = []
with open("semeval-tweets/twitter-test1.txt") as f:
  for x in f:
    dstest.append(x.split("\t"))

dftest = pd.DataFrame(data = dstest, columns = ["id", "sentiment", "tweet"])
dftest

,id,sentiment,tweet
0,163361196206957578,neutral,Candids: Heading to the Chateau Marmont in Wes...
1,768006053969268950,negative,@Dont__KAY_me omg same I was reading it in sch...
2,742616104384772304,neutral,Watching MTV Hits! The Wanted Chasing the sun!\n
3,102313285628711403,neutral,"""Bing one-ups knowledge graph, hires Encyclopa..."
4,653274888624828198,neutral,"""On Thursday, concealed-carry gun license hold..."
...,...,...,...
3526,897427103459983282,positive,"""@heavyoilcountry right you are, they could en..."
3527,494418545323766863,neutral,We will have to think about continuing in UPA:...
3528,556020888361859268,positive,ladies come throw that ass and niggas come thr...
3529,683639126147064420,neutral,@MandiieeKillaaa are you going to laidback Luk...


In [75]:
print(df["sentiment"].value_counts(dropna=False))

sentiment
neutral     919
positive    703
negative    378
Name: count, dtype: int64


### Preprocess

In [76]:
# lower case everything
df["tweet"] = df['tweet'].str.lower()
dftest["tweet"] = dftest['tweet'].str.lower()
df

,id,sentiment,tweet
0,262696992176304465,neutral,@oirisheye hey you! i'm gonna be in dublin in ...
1,410734138242126311,positive,literally so excited i'm going to a sam smith ...
2,237615985571058688,neutral,@windmobile will there be an option to buy the...
3,090473590077188360,neutral,our little ms. philippines. 👏👏👏#littlemsphilip...
4,450236582392850660,negative,"@angryraiderfan i know. this, tpp, expanded wa..."
...,...,...,...
1995,648550834532150634,positive,can it just be 5:30 tomorrow evening already b...
1996,022324697041215693,neutral,look for me on @gma tomorrow at 7:34am talking...
1997,885111074317866321,neutral,@theshannonburke after @immandaschmidt's call ...
1998,691260170402024265,neutral,"""taking go set a watchman with me to the beach..."


In [77]:
# remove punctuation
df["tweet"] = df['tweet'].str.replace(r'[^\w\s]', '', regex=True)
dftest["tweet"] = dftest['tweet'].str.replace(r'[^\w\s]', '', regex=True)
df

,id,sentiment,tweet
0,262696992176304465,neutral,oirisheye hey you im gonna be in dublin in feb...
1,410734138242126311,positive,literally so excited im going to a sam smith c...
2,237615985571058688,neutral,windmobile will there be an option to buy the ...
3,090473590077188360,neutral,our little ms philippines littlemsphilippines ...
4,450236582392850660,negative,angryraiderfan i know this tpp expanded wars a...
...,...,...,...
1995,648550834532150634,positive,can it just be 530 tomorrow evening already be...
1996,022324697041215693,neutral,look for me on gma tomorrow at 734am talking a...
1997,885111074317866321,neutral,theshannonburke after immandaschmidts call i f...
1998,691260170402024265,neutral,taking go set a watchman with me to the beach ...


### N-Gram and then BoW the dataset

In [78]:
vectoriser = skl.feature_extraction.text.TfidfVectorizer(ngram_range = (1,2))
Xtrain, Ytrain = vectoriser.fit_transform(df["tweet"]), df["sentiment"]
Ytrain = Ytrain.replace({"positive": 2, "neutral": 1, "negative": 0}).astype(int)
print(Xtrain.shape)
Ytrain

(2000, 34985)


0       1
1       2
2       1
3       1
4       0
       ..
1995    2
1996    1
1997    1
1998    1
1999    1
Name: sentiment, Length: 2000, dtype: int64

In [79]:
Xtest, Ytest = vectoriser.transform(dftest["tweet"]), dftest["sentiment"]
Ytest = Ytest.replace({"positive": 2, "neutral": 1, "negative": 0}).astype(int)
print(Xtrain.shape)
Ytest

(2000, 34985)


0       1
1       0
2       1
3       1
4       1
       ..
3526    2
3527    1
3528    2
3529    1
3530    1
Name: sentiment, Length: 3531, dtype: int64

# Classifier 1: SVM

In [80]:
model = skl.naive_bayes.MultinomialNB()
model.fit(Xtrain, Ytrain)
ypreds = model.predict(Xtest)
skl.metrics.accuracy_score(Ytest, ypreds)

0.5063721325403568